In [1]:
from google.colab import drive
drive.mount('/content/drive')

%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

import pandas as pd

COLUMNAS = ['loan_amnt','term','int_rate','installment','grade','sub_grade',
            'emp_length','home_ownership','annual_inc','verification_status',
            'issue_d','loan_status','purpose','addr_state','dti','open_acc',
            'pub_rec','revol_bal','revol_util','total_acc','mort_acc','pub_rec_bankruptcies']

df = pd.read_csv('/content/drive/MyDrive/Master UCLM/IA/accepted_2007_to_2018Q4.csv',
                 usecols=COLUMNAS, low_memory=False)

from sqlalchemy import create_engine
engine = create_engine('sqlite:///lending.db')
df.to_sql('loans', engine, if_exists='replace', index=False)

%sql sqlite:///lending.db
print(f'Listo  {len(df):,} filas | {df.shape[1]} columnas')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Listo  2,260,701 filas | 22 columnas


In [13]:
%%sql
SELECT loan_status, COUNT(*) AS total
FROM loans
GROUP BY loan_status
ORDER BY total DESC

 * sqlite:///lending.db
Done.


loan_status,total
Fully Paid,1076751
Current,878317
Charged Off,268559
Late (31-120 days),21467
In Grace Period,8436
Late (16-30 days),4349
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,40
None,33


In [14]:
%%sql
DROP VIEW IF EXISTS loans_clean;

 * sqlite:///lending.db
Done.


[]

**Data Cleaning & Preprocessing**

Target variable: Kept only Fully Paid and Charged Off loans. Excluded Current, Late, and other statuses as their final outcome is unknown.
Time window: Filtered loans issued from 2010 onwards to exclude data affected by the 2008 financial crisis, which introduced abnormal borrowing and default patterns.

Missing values: Removed all rows with null values in any of the selected features.
Feature selection: Kept only the 22 variables relevant to default prediction as defined in the business understanding phase.

In [15]:
%%sql
CREATE VIEW loans_clean AS
SELECT *
FROM loans
WHERE loan_status IN ('Fully Paid', 'Charged Off')
  AND CAST(SUBSTR(issue_d, 5, 4) AS INTEGER) >= 2010
  AND loan_amnt IS NOT NULL
  AND term IS NOT NULL
  AND int_rate IS NOT NULL
  AND installment IS NOT NULL
  AND grade IS NOT NULL
  AND emp_length IS NOT NULL
  AND home_ownership IS NOT NULL
  AND annual_inc IS NOT NULL
  AND verification_status IS NOT NULL
  AND purpose IS NOT NULL
  AND addr_state IS NOT NULL
  AND dti IS NOT NULL
  AND open_acc IS NOT NULL
  AND pub_rec IS NOT NULL
  AND revol_bal IS NOT NULL
  AND revol_util IS NOT NULL
  AND total_acc IS NOT NULL
  AND mort_acc IS NOT NULL
  AND pub_rec_bankruptcies IS NOT NULL

 * sqlite:///lending.db
Done.


[]

In [16]:
%%sql
SELECT COUNT(*) as total FROM loans_clean

 * sqlite:///lending.db
Done.


total
1220092


In [17]:
%%sql
SELECT loan_status, COUNT(*) as total FROM loans_clean WHERE loan_status IN ('Fully Paid', 'Charged Off') GROUP BY loan_status

 * sqlite:///lending.db
Done.


loan_status,total
Charged Off,240673
Fully Paid,979419


*grade*

In [19]:
%%sql
SELECT
    grade,
    COUNT(*) as total,
    SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) as defaults,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) / COUNT(*), 2) as tasa_default_pct
FROM loans_clean
GROUP BY grade
ORDER BY grade

 * sqlite:///lending.db
Done.


grade,total,defaults,tasa_default_pct
A,210557,12068,5.73
B,355792,46246,13.0
C,349459,77267,22.11
D,181826,54869,30.18
E,84975,32819,38.62
F,29171,13223,45.33
G,8312,4181,50.3


Finding: A clear relationship exists between grade and default rate. Grade A loans show only a 5.7% default rate, while Grade G reaches 50.3%, meaning 1 in 2 borrowers defaults. This confirms that grade is a strong predictor of default.

*interest rate*

In [20]:
%%sql
SELECT
    grade,
    ROUND(AVG(int_rate), 2) as avg_interest_rate,
    ROUND(AVG(CASE WHEN loan_status = 'Charged Off' THEN int_rate END), 2) as irate_default,
    ROUND(AVG(CASE WHEN loan_status = 'Fully Paid' THEN int_rate END), 2) as irate_paid
FROM loans_clean
GROUP BY grade
ORDER BY grade

 * sqlite:///lending.db
Done.


grade,avg_interest_rate,irate_default,irate_paid
A,7.1,7.39,7.08
B,10.66,10.78,10.65
C,14.03,14.09,14.02
D,17.78,17.82,17.76
E,21.24,21.25,21.24
F,25.11,25.21,25.03
G,27.97,28.09,27.84


Finding: Defaulted loans carry a slightly higher interest rate than paid ones within the same grade, but differences are minimal (decimals). This suggests that interest rate alone is not a strong differentiator, the grade itself captures the risk better.

*Purpose*

In [21]:
%%sql
SELECT
    purpose,
    COUNT(*) as total,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) / COUNT(*), 2) as default_rate_pct
FROM loans_clean
GROUP BY purpose
ORDER BY default_rate_pct DESC

 * sqlite:///lending.db
Done.


purpose,total,default_rate_pct
small_business,12835,30.16
renewable_energy,767,24.25
moving,8264,23.31
house,6367,22.04
medical,13477,21.15
other,67852,20.91
debt_consolidation,715066,20.9
major_purchase,25427,19.01
vacation,7969,18.99
home_improvement,77907,17.49


Finding: Small business loans have the highest default rate at 30.16%, followed by renewable energy (24.25%) and moving (23.31%). This suggests that loan purpose is a relevant risk indicator, with business-related and life-transition loans being the riskiest.

*DTI = Debt-to-Income ratio/income*

In [22]:
%%sql
SELECT
    loan_status,
    ROUND(AVG(dti), 2) as avg_dti,
    ROUND(AVG(annual_inc), 2) as avg_annual_income,
    ROUND(AVG(loan_amnt), 2) as avg_loan_amount
FROM loans_clean
GROUP BY loan_status

 * sqlite:///lending.db
Done.


loan_status,avg_dti,avg_annual_income,avg_loan_amount
Charged Off,20.14,72675.06,15959.77
Fully Paid,17.85,79553.28,14405.42


Finding: Defaulted borrowers show a higher DTI (20.14 vs 17.85), lower annual income ($72k vs $79k) and request larger loan amounts ($15.9 k  vs $14.4k ). This combination of more debt, less income and higher borrowing is a strong signal of default risk.

In [23]:
%%sql
SELECT
    emp_length,
    COUNT(*) as total,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) / COUNT(*), 2) as default_rate_pct
FROM loans_clean
GROUP BY emp_length
ORDER BY emp_length

 * sqlite:///lending.db
Done.


emp_length,total,default_rate_pct
1 year,84645,20.86
10+ years,431106,18.86
2 years,116571,20.1
3 years,102817,20.25
4 years,76432,20.04
5 years,80175,19.88
6 years,59928,19.6
7 years,57413,19.63
8 years,58904,20.09
9 years,49385,20.1


Finding: Employment length shows no significant impact on default rate. All categories hover around 19-21%, suggesting that how long someone has been employed is not a strong predictor of default in this dataset

In [24]:
df_clean = %sql SELECT * FROM loans_clean
df_clean = df_clean.DataFrame()
df_clean.to_csv('/content/drive/MyDrive/Master UCLM/IA/lending_club_clean.csv', index=False)
print(f'Guardado  {len(df_clean):,} filas')

 * sqlite:///lending.db
Done.
Guardado ✅ 1,220,092 filas
